# V8.26 Zero-Sum Extinction — Full Pipeline

Reproduces the winning submission (`v8_26_zero_sum_extinction.csv`, LB **653 725.86**).

**Pipeline stages:**
- **A** — STL + LightGBM base forecast (aggregate daily Revenue / COGS)
- **F** — Flat ×1.04 ladder anchor (v8_12)
- **D** — L-BFGS-B smooth-MAE calibration toward anchor (v8_14)
- **H** — Amplify Δ: `y = anchor + 2·(calibrated − anchor)` (v8_20)
- **G** — Closed-form Ridge per-month multiplier correction (v8_21)
- **I** — Dead-SKU mining from `order_items` (m37)
- **J** — Zero-sum extinction + revenue-preserving rescale (v8_26)

In [ ]:
# Stage 0 — install / upgrade required packages
import subprocess, sys

pkgs = [
    "lightgbm>=4.0",
    "statsmodels>=0.14",
    "scikit-learn>=1.3",
    "scipy>=1.10",
    "shap>=0.44",
    "pandas>=2.0",
    "numpy>=1.24",
    "matplotlib>=3.7",
    "seaborn>=0.12",
]

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + pkgs,
    check=True,
)
print("All packages OK")

In [ ]:
# Stage 0b — imports and global config
import warnings
warnings.filterwarnings("ignore")

import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.optimize import minimize
from scipy.linalg import solve

from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import lightgbm as lgb

from statsmodels.tsa.seasonal import STL

# ── paths ────────────────────────────────────────────────────────────
ROOT = Path("d:/DATATHON-2026-GenApLucUWU")
RAW  = ROOT / "data" / "raw"
OUT  = ROOT / "data" / "output"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

print(f"ROOT : {ROOT}")
print(f"RAW  : {RAW}")
print(f"OUT  : {OUT}")

In [ ]:
# Stage 0c — load all raw data
sales       = pd.read_csv(RAW / "sales.csv",          parse_dates=["Date"])
sample_sub  = pd.read_csv(RAW / "sample_submission.csv", parse_dates=["Date"])
orders      = pd.read_csv(RAW / "orders.csv",         parse_dates=["order_date"])
order_items = pd.read_csv(RAW / "order_items.csv")
products    = pd.read_csv(RAW / "products.csv")
promotions  = pd.read_csv(RAW / "promotions.csv",     parse_dates=["start_date", "end_date"])
web_traffic = pd.read_csv(RAW / "web_traffic.csv",    parse_dates=["date"])
inventory   = pd.read_csv(RAW / "inventory.csv",      parse_dates=["snapshot_date"])

sales.sort_values("Date", inplace=True)
sales.reset_index(drop=True, inplace=True)

TEST_DATES = sample_sub["Date"].values
TRAIN_END  = pd.Timestamp("2022-12-31")
TEST_START = pd.Timestamp("2023-01-01")

print(f"Train: {sales['Date'].min().date()} → {sales['Date'].max().date()}  ({len(sales)} rows)")
print(f"Test : {sample_sub['Date'].min().date()} → {sample_sub['Date'].max().date()}  ({len(sample_sub)} rows)")
print(f"Products: {len(products)}  |  Orders: {len(orders)}  |  OrderItems: {len(order_items)}")

## Stage A — STL + LightGBM Base Forecast

In [ ]:
# Stage A-1 — calendar & external features builder
def make_calendar_features(df: pd.DataFrame, date_col: str = "Date") -> pd.DataFrame:
    """Add rich calendar features to a date-indexed frame."""
    d = df.copy()
    dt = d[date_col]
    d["dow"]       = dt.dt.dayofweek          # 0=Mon … 6=Sun
    d["dom"]       = dt.dt.day
    d["month"]     = dt.dt.month
    d["quarter"]   = dt.dt.quarter
    d["doy"]       = dt.dt.dayofyear
    d["week"]      = dt.dt.isocalendar().week.astype(int)
    d["year"]      = dt.dt.year
    d["is_weekend"]    = (d["dow"] >= 5).astype(int)
    d["is_month_start"] = (d["dom"] <= 3).astype(int)
    d["is_month_end"]   = (d["dom"] >= 28).astype(int)
    d["dte"]       = (dt + pd.offsets.MonthEnd(0) - dt).dt.days  # days to end of month

    # Vietnamese public holidays (approximate calendar flags)
    is_tet = (
        (d["month"] == 1) & (d["dom"].between(20, 31)) |
        (d["month"] == 2) & (d["dom"].between(1, 10))
    )
    d["is_tet_window"] = is_tet.astype(int)
    # Sin/cos encoding for periodicity
    d["sin_dow"]   = np.sin(2 * np.pi * d["dow"]   / 7)
    d["cos_dow"]   = np.cos(2 * np.pi * d["dow"]   / 7)
    d["sin_month"] = np.sin(2 * np.pi * d["month"] / 12)
    d["cos_month"] = np.cos(2 * np.pi * d["month"] / 12)
    d["sin_doy"]   = np.sin(2 * np.pi * d["doy"]   / 365)
    d["cos_doy"]   = np.cos(2 * np.pi * d["doy"]   / 365)
    return d


def add_web_traffic_features(df: pd.DataFrame, traffic: pd.DataFrame,
                              date_col_df: str = "Date",
                              date_col_tr: str = "date") -> pd.DataFrame:
    """Merge smoothed weekly web-traffic sessions into main frame."""
    tr = traffic.copy()
    tr["sessions_norm"] = tr["sessions"] / tr["sessions"].median()
    tr_daily = tr.groupby(date_col_tr)["sessions_norm"].mean().reset_index()
    tr_daily.columns = [date_col_df, "sessions_norm"]
    return df.merge(tr_daily, on=date_col_df, how="left")


def add_promo_features(df: pd.DataFrame, promos: pd.DataFrame,
                       date_col: str = "Date") -> pd.DataFrame:
    """Add a flag for whether a promotion is active on each date."""
    dates = df[date_col].values
    promo_active = np.zeros(len(dates), dtype=int)
    discount_intensity = np.zeros(len(dates), dtype=float)

    for _, row in promos.iterrows():
        mask = (dates >= row["start_date"]) & (dates <= row["end_date"])
        promo_active[mask] = 1
        dv = row["discount_value"] if pd.notna(row["discount_value"]) else 0
        discount_intensity[mask] += dv

    d = df.copy()
    d["promo_active"]      = promo_active
    d["discount_intensity"] = discount_intensity
    return d

print("Feature builder functions defined.")

In [ ]:
# Stage A-2 — STL decomposition on training Revenue
rev_series = sales.set_index("Date")["Revenue"]

# Fill any gaps with forward-fill (STL requires complete series)
full_idx = pd.date_range(rev_series.index.min(), rev_series.index.max(), freq="D")
rev_series = rev_series.reindex(full_idx).ffill()

# STL with weekly (7-day) seasonality
stl_rev = STL(rev_series, period=7, robust=True)
res_rev  = stl_rev.fit()

# Build a DataFrame with STL components (trend + seasonal)
stl_df = pd.DataFrame({
    "Date"      : rev_series.index,
    "stl_trend" : res_rev.trend.values,
    "stl_season": res_rev.seasonal.values,
    "stl_resid" : res_rev.resid.values,
})
stl_df["stl_baseline_rev"] = stl_df["stl_trend"] + stl_df["stl_season"]

# Same for COGS
cogs_series = sales.set_index("Date")["COGS"]
cogs_series = cogs_series.reindex(full_idx).ffill()

stl_cogs = STL(cogs_series, period=7, robust=True)
res_cogs  = stl_cogs.fit()

stl_df["stl_trend_cogs"]    = res_cogs.trend.values
stl_df["stl_season_cogs"]   = res_cogs.seasonal.values
stl_df["stl_baseline_cogs"] = stl_df["stl_trend_cogs"] + stl_df["stl_season_cogs"]

print(f"STL Revenue  — trend range: [{res_rev.trend.min():.0f}, {res_rev.trend.max():.0f}]")
print(f"STL COGS     — trend range: [{res_cogs.trend.min():.0f}, {res_cogs.trend.max():.0f}]")

In [ ]:
# Stage A-3 — build full feature table (train + test horizon)
# Create a full date index covering train + test
all_dates = pd.date_range(rev_series.index.min(), "2024-07-01", freq="D")
feat = pd.DataFrame({"Date": all_dates})

# Add calendar features
feat = make_calendar_features(feat, "Date")

# Merge web_traffic (available only during train)
feat = add_web_traffic_features(feat, web_traffic)
feat["sessions_norm"] = feat["sessions_norm"].fillna(feat["sessions_norm"].median())

# Merge promo features
feat = add_promo_features(feat, promotions)

# Merge Revenue / COGS targets (NaN for test rows)
feat = feat.merge(sales[["Date", "Revenue", "COGS"]], on="Date", how="left")

# Merge STL baselines (only on train range)
feat = feat.merge(stl_df[["Date", "stl_baseline_rev", "stl_baseline_cogs",
                            "stl_trend", "stl_season",
                            "stl_trend_cogs", "stl_season_cogs"]],
                  on="Date", how="left")

# ── Add lag features (using Revenue from train) ───────────────────────
# Sorted revenue series for lag computation
rev_full = feat.set_index("Date")["Revenue"]

for lag in [1, 2, 3, 7, 14, 28, 364, 365, 371]:
    feat[f"rev_lag_{lag}"] = rev_full.shift(lag).values

for win in [7, 14, 28, 91, 182, 364]:
    feat[f"rev_roll_mean_{win}"] = rev_full.shift(1).rolling(win, min_periods=1).mean().values
    feat[f"rev_roll_std_{win}"]  = rev_full.shift(1).rolling(win, min_periods=1).std().fillna(0).values

# Same for COGS
cogs_full = feat.set_index("Date")["COGS"]
for lag in [1, 7, 28, 364]:
    feat[f"cogs_lag_{lag}"] = cogs_full.shift(lag).values

print(f"Feature table shape: {feat.shape}")
print(f"Train rows: {feat[feat['Date'] <= TRAIN_END].shape[0]}")
print(f"Test  rows: {feat[feat['Date'] >= TEST_START].shape[0]}")

In [ ]:
# Stage A-4 — project STL baselines into test period
# We extrapolate the trend using the last 90-day linear slope and copy seasonal pattern

train_mask = feat["Date"] <= TRAIN_END
test_mask  = feat["Date"] >= TEST_START

# --- Revenue trend extrapolation ---
trend_train = feat.loc[train_mask, ["Date", "stl_trend"]].dropna().copy()
trend_train["t"] = (trend_train["Date"] - trend_train["Date"].iloc[-90]).dt.days

X_slope = trend_train["t"].iloc[-90:].values.reshape(-1, 1)
y_slope = trend_train["stl_trend"].iloc[-90:].values
slope_rev  = np.polyfit(X_slope.flatten(), y_slope, 1)

last_train_date = trend_train["Date"].iloc[-1]
test_dates = feat.loc[test_mask, "Date"]
t_test = (test_dates - last_train_date).dt.days.values
feat.loc[test_mask, "stl_trend"] = np.polyval(slope_rev, t_test)

# Clip trend to reasonable bounds (50% – 200% of last train trend)
last_trend_rev = trend_train["stl_trend"].iloc[-1]
feat.loc[test_mask, "stl_trend"] = feat.loc[test_mask, "stl_trend"].clip(
    last_trend_rev * 0.5, last_trend_rev * 2.0
)

# --- COGS trend extrapolation ---
trend_cogs = feat.loc[train_mask, ["Date", "stl_trend_cogs"]].dropna().copy()
X_slope_c = (trend_cogs["Date"].iloc[-90:] - last_train_date).dt.days.values
y_slope_c = trend_cogs["stl_trend_cogs"].iloc[-90:].values
slope_cogs = np.polyfit(X_slope_c, y_slope_c, 1)
feat.loc[test_mask, "stl_trend_cogs"] = np.polyval(slope_cogs, t_test)

last_trend_cogs = trend_cogs["stl_trend_cogs"].iloc[-1]
feat.loc[test_mask, "stl_trend_cogs"] = feat.loc[test_mask, "stl_trend_cogs"].clip(
    last_trend_cogs * 0.5, last_trend_cogs * 2.0
)

# --- Seasonal: copy same DOY from 2021/2022 average (nearest year) ---
# Build a DOW×month seasonal lookup from training data
season_lookup = (
    feat.loc[train_mask]
    .dropna(subset=["stl_season"])
    .groupby(["month", "dow"])["stl_season"].mean()
)
feat["_month_dow"] = list(zip(feat["month"], feat["dow"]))
feat["stl_season"] = feat["stl_season"].combine_first(
    feat["_month_dow"].map(season_lookup)
)
feat["stl_season"].fillna(0, inplace=True)

season_lookup_c = (
    feat.loc[train_mask]
    .dropna(subset=["stl_season_cogs"])
    .groupby(["month", "dow"])["stl_season_cogs"].mean()
)
feat["stl_season_cogs"] = feat["stl_season_cogs"].combine_first(
    feat["_month_dow"].map(season_lookup_c)
)
feat["stl_season_cogs"].fillna(0, inplace=True)

# Recompute baselines for test period
feat["stl_baseline_rev"]  = feat["stl_trend"]      + feat["stl_season"]
feat["stl_baseline_cogs"] = feat["stl_trend_cogs"] + feat["stl_season_cogs"]

# Drop temp column
feat.drop(columns=["_month_dow"], inplace=True)

print("STL baseline projected into test period.")
print(feat.loc[test_mask, ["Date", "stl_baseline_rev", "stl_baseline_cogs"]].head(5))

In [ ]:
# Stage A-5 — fill lag features for test period using recursive simulation
# Since test lag features need Revenue values that don't exist yet,
# we substitute with the STL baseline for lag > 364 and rolling windows.

# For lag 364/365/371 — copy from same day last year (available in train)
date_to_rev = sales.set_index("Date")["Revenue"].to_dict()
date_to_cogs = sales.set_index("Date")["COGS"].to_dict()

for idx, row in feat.loc[test_mask].iterrows():
    d = row["Date"]
    for lag in [364, 365, 371]:
        prev = d - pd.Timedelta(days=lag)
        if pd.isna(feat.at[idx, f"rev_lag_{lag}"]):
            feat.at[idx, f"rev_lag_{lag}"] = date_to_rev.get(prev, feat.at[idx, "stl_baseline_rev"])

# For short lags in test period — fill with STL baseline
for lag in [1, 2, 3, 7, 14, 28]:
    feat.loc[test_mask, f"rev_lag_{lag}"] = feat.loc[test_mask, f"rev_lag_{lag}"].fillna(
        feat.loc[test_mask, "stl_baseline_rev"]
    )

# Fill rolling features with STL baseline for test
for win in [7, 14, 28, 91, 182, 364]:
    feat.loc[test_mask, f"rev_roll_mean_{win}"] = feat.loc[test_mask, f"rev_roll_mean_{win}"].fillna(
        feat.loc[test_mask, "stl_baseline_rev"]
    )
    feat.loc[test_mask, f"rev_roll_std_{win}"] = feat.loc[test_mask, f"rev_roll_std_{win}"].fillna(0)

for lag in [1, 7, 28, 364]:
    feat.loc[test_mask, f"cogs_lag_{lag}"] = feat.loc[test_mask, f"cogs_lag_{lag}"].fillna(
        feat.loc[test_mask, "stl_baseline_cogs"]
    )

print("Lag/rolling features filled for test period.")
print(f"NaN count in features: {feat.loc[test_mask].isnull().sum().sum()}")

In [ ]:
# Stage A-6 — define feature columns and train LightGBM models

FEATURE_COLS = [
    "dow", "dom", "month", "quarter", "doy", "week", "year",
    "is_weekend", "is_month_start", "is_month_end", "dte",
    "is_tet_window",
    "sin_dow", "cos_dow", "sin_month", "cos_month", "sin_doy", "cos_doy",
    "sessions_norm", "promo_active", "discount_intensity",
    "stl_baseline_rev", "stl_trend", "stl_season",
    "rev_lag_1", "rev_lag_2", "rev_lag_3",
    "rev_lag_7", "rev_lag_14", "rev_lag_28",
    "rev_lag_364", "rev_lag_365", "rev_lag_371",
    "rev_roll_mean_7", "rev_roll_mean_14", "rev_roll_mean_28",
    "rev_roll_mean_91", "rev_roll_mean_182", "rev_roll_mean_364",
    "rev_roll_std_7", "rev_roll_std_28",
]

COGS_FEATURE_COLS = [
    "dow", "dom", "month", "quarter", "doy", "week", "year",
    "is_weekend", "is_month_start", "is_month_end", "dte",
    "is_tet_window",
    "sin_dow", "cos_dow", "sin_month", "cos_month", "sin_doy", "cos_doy",
    "sessions_norm", "promo_active", "discount_intensity",
    "stl_baseline_cogs", "stl_trend_cogs", "stl_season_cogs",
    "cogs_lag_1", "cogs_lag_7", "cogs_lag_28", "cogs_lag_364",
    # Cross-feature: revenue prediction as proxy
    "stl_baseline_rev",
    "rev_lag_364", "rev_lag_365",
]

# Training set: rows with valid Revenue target and no NaN in features
train_df = feat.loc[train_mask].copy()

# Drop rows with too many NaN (first ~400 rows lack lag_364)
train_df_rev  = train_df.dropna(subset=FEATURE_COLS + ["Revenue"])
train_df_cogs = train_df.dropna(subset=COGS_FEATURE_COLS + ["COGS"])

X_rev  = train_df_rev[FEATURE_COLS].values
y_rev  = train_df_rev["Revenue"].values

X_cogs = train_df_cogs[COGS_FEATURE_COLS].values
y_cogs = train_df_cogs["COGS"].values

print(f"Revenue  training rows: {len(X_rev)}")
print(f"COGS     training rows: {len(X_cogs)}")

In [ ]:
# Stage A-7 — LightGBM Revenue model with 5-fold expanding TimeSeriesSplit CV

LGBM_PARAMS = dict(
    objective="regression_l1",   # MAE loss
    metric="mae",
    n_estimators=1200,
    learning_rate=0.03,
    num_leaves=63,
    max_depth=8,
    min_child_samples=30,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    reg_alpha=0.1,
    reg_lambda=0.5,
    random_state=SEED,
    verbose=-1,
    n_jobs=-1,
)

tscv = TimeSeriesSplit(n_splits=5, gap=14)

cv_maes = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_rev)):
    model_cv = lgb.LGBMRegressor(**LGBM_PARAMS)
    model_cv.fit(X_rev[tr_idx], y_rev[tr_idx],
                 eval_set=[(X_rev[val_idx], y_rev[val_idx])],
                 callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds_val = model_cv.predict(X_rev[val_idx])
    mae = mean_absolute_error(y_rev[val_idx], preds_val)
    cv_maes.append(mae)
    print(f"  Fold {fold+1}: MAE = {mae:,.0f}")

print(f"\nCV MAE Revenue: {np.mean(cv_maes):,.0f} ± {np.std(cv_maes):,.0f}")

# Retrain on full training set
rev_model = lgb.LGBMRegressor(**LGBM_PARAMS)
rev_model.fit(X_rev, y_rev, callbacks=[lgb.log_evaluation(-1)])
print("Revenue LightGBM trained on full train set.")

In [ ]:
# Stage A-8 — LightGBM COGS model

LGBM_PARAMS_COGS = LGBM_PARAMS.copy()

cv_maes_cogs = []
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_cogs)):
    model_cv = lgb.LGBMRegressor(**LGBM_PARAMS_COGS)
    model_cv.fit(X_cogs[tr_idx], y_cogs[tr_idx],
                 eval_set=[(X_cogs[val_idx], y_cogs[val_idx])],
                 callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)])
    preds_val = model_cv.predict(X_cogs[val_idx])
    mae = mean_absolute_error(y_cogs[val_idx], preds_val)
    cv_maes_cogs.append(mae)
    print(f"  Fold {fold+1}: MAE = {mae:,.0f}")

print(f"\nCV MAE COGS: {np.mean(cv_maes_cogs):,.0f} ± {np.std(cv_maes_cogs):,.0f}")

cogs_model = lgb.LGBMRegressor(**LGBM_PARAMS_COGS)
cogs_model.fit(X_cogs, y_cogs, callbacks=[lgb.log_evaluation(-1)])
print("COGS LightGBM trained on full train set.")

In [ ]:
# Stage A-9 — generate Stage A forecast for test horizon

test_df = feat.loc[test_mask].copy()

X_test_rev  = test_df[FEATURE_COLS].fillna(0).values
X_test_cogs = test_df[COGS_FEATURE_COLS].fillna(0).values

stage_a_rev  = rev_model.predict(X_test_rev)
stage_a_cogs = cogs_model.predict(X_test_cogs)

# Hard-clip to positive values
stage_a_rev  = np.clip(stage_a_rev,  0, None)
stage_a_cogs = np.clip(stage_a_cogs, 0, None)

stage_a = pd.DataFrame({
    "Date"   : test_df["Date"].values,
    "Revenue": stage_a_rev,
    "COGS"   : stage_a_cogs,
})

print("Stage A forecast generated.")
print(f"Revenue — mean: {stage_a_rev.mean():,.0f}  | min: {stage_a_rev.min():,.0f}  | max: {stage_a_rev.max():,.0f}")
print(f"COGS    — mean: {stage_a_cogs.mean():,.0f}  | min: {stage_a_cogs.min():,.0f}  | max: {stage_a_cogs.max():,.0f}")
print(stage_a.head(5))

## Stage F — Flat ×1.04 Ladder (v8_12)

In [ ]:
# Stage F — apply global ×1.04 multiplier to Stage A output
LADDER_MULT = 1.04

v8_12 = stage_a.copy()
v8_12["Revenue"] = stage_a["Revenue"] * LADDER_MULT
v8_12["COGS"]    = stage_a["COGS"]    * LADDER_MULT

print(f"Stage F (×{LADDER_MULT}) applied.")
print(f"Revenue — mean: {v8_12['Revenue'].mean():,.0f}")

## Stage D — L-BFGS-B Smooth-MAE Calibration (v8_14)

In [ ]:
# Stage D — Smooth-MAE optimisation bounded to anchor ± 20%
# Minimise: L = Σ|y - y_teacher| + μ·Σ(y - y_metric_ref)²
# y_teacher    = v8_12 (flat anchor)
# y_metric_ref = Stage A (original forecast)

def smooth_mae_loss(y, y_teacher, y_metric_ref, mu):
    """Smooth L1 + L2 quadratic regularisation."""
    delta = 1.0
    diff  = y - y_teacher
    # Huber-like smooth MAE
    huber = np.where(
        np.abs(diff) <= delta,
        0.5 * diff ** 2 / delta,
        np.abs(diff) - 0.5 * delta,
    )
    reg = np.sum((y - y_metric_ref) ** 2)
    return np.sum(huber) + mu * reg


def smooth_mae_grad(y, y_teacher, y_metric_ref, mu):
    delta = 1.0
    diff  = y - y_teacher
    huber_grad = np.where(np.abs(diff) <= delta, diff / delta, np.sign(diff))
    return huber_grad + 2 * mu * (y - y_metric_ref)


def calibrate_lbfgsb(y_init, y_teacher, y_metric_ref, mu, clip_frac=0.20):
    """Run L-BFGS-B with box constraints y_teacher ± clip_frac."""
    lb = y_teacher * (1.0 - clip_frac)
    ub = y_teacher * (1.0 + clip_frac)
    bounds = list(zip(lb, ub))

    result = minimize(
        fun=lambda y: smooth_mae_loss(y, y_teacher, y_metric_ref, mu),
        x0=y_init.copy(),
        jac=lambda y: smooth_mae_grad(y, y_teacher, y_metric_ref, mu),
        method="L-BFGS-B",
        bounds=bounds,
        options={"maxiter": 500, "ftol": 1e-12},
    )
    return result.x


# Sweep μ values and pick the best (simplified: use fixed μ=1e-7)
MU_VALUES = [1e-7, 3e-7, 1e-6]

y_init        = stage_a["Revenue"].values.copy()
y_teacher_rev = v8_12["Revenue"].values
y_metric_rev  = stage_a["Revenue"].values

calibrated_rev_candidates = []
for mu in MU_VALUES:
    cal = calibrate_lbfgsb(y_init, y_teacher_rev, y_metric_rev, mu)
    calibrated_rev_candidates.append(cal)
    diff_from_anchor = np.mean(np.abs(cal - y_teacher_rev))
    print(f"  μ={mu:.0e}: mean |y - anchor| = {diff_from_anchor:,.0f}")

# Pick μ=1e-7 (stays closest to anchor which scores better per surrogate)
v8_14_rev = calibrated_rev_candidates[0]

# COGS: same calibration
v8_14_cogs = calibrate_lbfgsb(
    stage_a["COGS"].values.copy(),
    v8_12["COGS"].values,
    stage_a["COGS"].values,
    mu=1e-7,
)

v8_14 = pd.DataFrame({
    "Date"   : stage_a["Date"].values,
    "Revenue": np.clip(v8_14_rev,  0, None),
    "COGS"   : np.clip(v8_14_cogs, 0, None),
})
print("\nStage D calibrated.")
print(f"Revenue mean: {v8_14['Revenue'].mean():,.0f}")

## Stage H — Amplify Δ (v8_20)

In [ ]:
# Stage H — y = y_base + α·(y_calibrated − y_base),  α=2.0
ALPHA = 2.0

v8_20_rev  = v8_12["Revenue"].values + ALPHA * (v8_14["Revenue"].values - v8_12["Revenue"].values)
v8_20_cogs = v8_12["COGS"].values    + ALPHA * (v8_14["COGS"].values    - v8_12["COGS"].values)

v8_20 = pd.DataFrame({
    "Date"   : stage_a["Date"].values,
    "Revenue": np.clip(v8_20_rev,  0, None),
    "COGS"   : np.clip(v8_20_cogs, 0, None),
})
print(f"Stage H (α={ALPHA}): Revenue mean = {v8_20['Revenue'].mean():,.0f}")

## Stage G — Segmented Month Regression (v8_21)

In [ ]:
# Stage G — closed-form Ridge per-month multiplier correction
# We model each month's adjustment using Ridge over the train Revenue monthly means
# vs the Stage A monthly means on training data.

# Compute monthly means from training data
train_monthly = (
    sales
    .assign(month=sales["Date"].dt.month)
    .groupby("month")["Revenue"].mean()
    .reindex(range(1, 13))
)

# Compute Stage-A monthly means on training period (in-sample prediction)
stage_a_in_sample = pd.DataFrame({
    "Date"   : feat.loc[train_df_rev.index, "Date"].values,
    "Revenue": rev_model.predict(X_rev),
})
stage_a_in_sample["month"] = stage_a_in_sample["Date"].dt.month
stageA_monthly = stage_a_in_sample.groupby("month")["Revenue"].mean()

# Per-month ratio: actual / predicted, regularised with γ=2e-5, clipped [0.98, 1.02]
GAMMA = 2e-5
month_ratios = {}
for m in range(1, 13):
    actual = train_monthly.get(m, np.nan)
    pred   = stageA_monthly.get(m, np.nan)
    if pd.isna(actual) or pd.isna(pred) or pred == 0:
        ratio = 1.0
    else:
        # Closed-form Ridge: p* = (X'X + λI)^{-1} X'y = actual/(pred + γ)
        ratio = actual / (pred + GAMMA * pred)
    # Clip to [0.98, 1.02]
    month_ratios[m] = float(np.clip(ratio, 0.98, 1.02))

print("Per-month multipliers (Stage G):")
for m, r in month_ratios.items():
    print(f"  Month {m:2d}: {r:.5f}")

# Apply per-month multipliers to v8_20
v8_21 = v8_20.copy()
months_test = pd.DatetimeIndex(v8_21["Date"]).month

v8_21["Revenue"] = v8_21["Revenue"] * months_test.map(month_ratios)
v8_21["COGS"]    = v8_21["COGS"]    * months_test.map(month_ratios)

print(f"\nStage G applied: Revenue mean = {v8_21['Revenue'].mean():,.0f}")

## Stage I — Dead-SKU Mining (m37)

In [ ]:
# Stage I — Mine dead SKUs from order_items × orders × products
# Window: 2019-01-01 → 2022-12-31 (training period)

TRAIN_START_M37 = "2019-01-01"
TRAIN_END_M37   = "2022-12-31"

# Merge order_items with orders to get order_date
oi_orders = order_items.merge(
    orders[["order_id", "order_date", "order_status"]],
    on="order_id",
    how="inner",
)

# Drop returned orders
oi_orders = oi_orders[oi_orders["order_status"] != "returned"].copy()

# Filter to training window
oi_orders = oi_orders[
    (oi_orders["order_date"] >= TRAIN_START_M37) &
    (oi_orders["order_date"] <= TRAIN_END_M37)
].copy()

# Compute line gross
oi_orders["line_gross"] = (
    oi_orders["quantity"] * oi_orders["unit_price"] - oi_orders["discount_amount"]
)
oi_orders["year"] = oi_orders["order_date"].dt.year

# Pivot to (product_id × year) revenue
piv = (
    oi_orders.groupby(["product_id", "year"])["line_gross"]
    .sum()
    .unstack("year")
    .fillna(0)
)

# Ensure 2021 and 2022 columns exist
for yr in [2021, 2022]:
    if yr not in piv.columns:
        piv[yr] = 0.0

piv["yoy_2021_2022"] = (piv[2022] - piv[2021]) / piv[2021].clip(lower=1e-6)

# Adaptive quantile: find dead SKUs with YoY ≤ -25%
# Start at 99.5th percentile of 2021 revenue; relax until ≥5 SKUs found
quantile_steps = [0.995, 0.99, 0.985, 0.98, 0.975, 0.97]
dead_skus = pd.DataFrame()

for q in quantile_steps:
    threshold_2021 = piv[2021].quantile(q)
    candidates = piv[
        (piv[2021] >= threshold_2021) & (piv["yoy_2021_2022"] <= -0.25)
    ]
    if len(candidates) >= 5:
        dead_skus = candidates.reset_index()
        print(f"Found {len(dead_skus)} dead SKUs at quantile={q:.3f}")
        break

if dead_skus.empty:
    print("No dead SKUs found — using all SKUs with YoY ≤ -50%")
    dead_skus = piv[piv["yoy_2021_2022"] <= -0.50].reset_index().head(5)

# Merge with product names
dead_skus = dead_skus.merge(
    products[["product_id", "product_name", "category"]],
    on="product_id",
    how="left",
)

print("\nDead SKUs:")
display_cols = ["product_id", "product_name", "category", 2021, 2022, "yoy_2021_2022"]
display_cols = [c for c in display_cols if c in dead_skus.columns]
print(dead_skus[display_cols].to_string(index=False))

In [ ]:
# Stage I-b — compute dead-weight by calendar month

dead_product_ids = set(dead_skus["product_id"].tolist())

oi_orders["month"] = oi_orders["order_date"].dt.month

# Pool 2019 + 2022 line-gross by month
monthly_pool = oi_orders[oi_orders["year"].isin([2019, 2022])].copy()

total_by_month = monthly_pool.groupby("month")["line_gross"].sum()
dead_by_month  = (
    monthly_pool[monthly_pool["product_id"].isin(dead_product_ids)]
    .groupby("month")["line_gross"]
    .sum()
)

dead_weight = (dead_by_month / total_by_month).reindex(range(1, 13)).fillna(0.0)

print("Dead-weight by month:")
for m, w in dead_weight.items():
    print(f"  Month {m:2d}: {w:.4f} ({w*100:.2f}%)")

# Save to JSON for reference
dead_weight_json = {str(k): float(v) for k, v in dead_weight.items()}
with open(OUT / "dead_weight_by_month.json", "w") as f:
    json.dump(dead_weight_json, f, indent=2)
print("\nSaved dead_weight_by_month.json")

## Stage J — Zero-Sum Extinction (v8_26)

In [ ]:
# Stage J — extinction penalty + revenue-preserving rescale
#
# For each row with calendar month m:
#   factor[m] = clip(1 − r·dead_weight[m], 0, 1)   # extinction penalty, r=0.70
#   rev_ext   = rev_base * factor[m]
#   K         = sum_orig / sum_ext                  # global compensation
#   rev_final = rev_ext * K
#   cogs_final= cogs_ext * K  (ratio preserved)

EXTINCTION_RATE = 0.70

months_v8_21 = pd.DatetimeIndex(v8_21["Date"]).month

# Compute extinction factor per row
factors = np.array([
    float(np.clip(1.0 - EXTINCTION_RATE * dead_weight.get(m, 0.0), 0.0, 1.0))
    for m in months_v8_21
])

rev_base = v8_21["Revenue"].values
cogs_base = v8_21["COGS"].values

rev_ext  = rev_base  * factors
cogs_ext = cogs_base * factors

sum_orig_rev  = rev_base.sum()
sum_ext_rev   = rev_ext.sum()
K_rev         = sum_orig_rev / sum_ext_rev if sum_ext_rev > 0 else 1.0

sum_orig_cogs = cogs_base.sum()
sum_ext_cogs  = cogs_ext.sum()
K_cogs        = sum_orig_cogs / sum_ext_cogs if sum_ext_cogs > 0 else 1.0

rev_final  = rev_ext  * K_rev
cogs_final = cogs_ext * K_cogs

print(f"Extinction rate r = {EXTINCTION_RATE}")
print(f"K_rev  = {K_rev:.7f}")
print(f"K_cogs = {K_cogs:.7f}")
print(f"Rev  — before: {sum_orig_rev:,.2f}  |  after: {rev_final.sum():,.2f}")
print(f"COGS — before: {sum_orig_cogs:,.2f}  |  after: {cogs_final.sum():,.2f}")
print(f"Factors range: [{factors.min():.4f}, {factors.max():.4f}]")

In [ ]:
# Stage J-b — assemble final submission and save

v8_26 = pd.DataFrame({
    "Date"   : v8_21["Date"].values,
    "Revenue": np.round(rev_final,  2),
    "COGS"   : np.round(cogs_final, 2),
})

# Align to sample_submission row order
submission = sample_sub[["Date"]].merge(v8_26, on="Date", how="left")

# Sanity check
assert len(submission) == len(sample_sub), "Row count mismatch vs sample_submission!"
assert submission["Revenue"].isna().sum() == 0, "NaN Revenue in submission!"
assert submission["COGS"].isna().sum()    == 0, "NaN COGS in submission!"

out_path = OUT / "v8_26_zero_sum_extinction.csv"
submission.to_csv(out_path, index=False)
print(f"Submission saved → {out_path}")
print(f"Rows: {len(submission)}  |  Columns: {list(submission.columns)}")
print(submission.head(5))

## Validation Metrics & Visualisations

In [ ]:
# Validation metrics on training set (in-sample)

y_train_true = y_rev
y_train_pred = rev_model.predict(X_rev)

mae_train  = mean_absolute_error(y_train_true, y_train_pred)
rmse_train = mean_squared_error(y_train_true, y_train_pred, squared=False)
r2_train   = r2_score(y_train_true, y_train_pred)

print("=== In-sample Revenue metrics ===")
print(f"  MAE  : {mae_train:>12,.2f}")
print(f"  RMSE : {rmse_train:>12,.2f}")
print(f"  R²   : {r2_train:>12.6f}")

print("\n=== Cross-validation Revenue MAE ===")
print(f"  CV-MAE mean : {np.mean(cv_maes):>12,.2f}")
print(f"  CV-MAE std  : {np.std(cv_maes):>12,.2f}")

print("\n=== Cross-validation COGS MAE ===")
print(f"  CV-MAE mean : {np.mean(cv_maes_cogs):>12,.2f}")
print(f"  CV-MAE std  : {np.std(cv_maes_cogs):>12,.2f}")

In [ ]:
# Forecast visualisation
fig, axes = plt.subplots(2, 1, figsize=(16, 8), sharex=False)

# Training history (last 2 years)
sales_plot = sales[sales["Date"] >= "2021-01-01"]

axes[0].plot(sales_plot["Date"], sales_plot["Revenue"] / 1e6,
             label="Actual (train)", color="steelblue", linewidth=1.2)
axes[0].plot(submission["Date"], submission["Revenue"] / 1e6,
             label="V8.26 Forecast", color="tomato", linewidth=1.2)
axes[0].set_title("Revenue Forecast — V8.26 Zero-Sum Extinction", fontsize=13)
axes[0].set_ylabel("Revenue (M VND)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(sales_plot["Date"], sales_plot["COGS"] / 1e6,
             label="Actual (train)", color="steelblue", linewidth=1.2)
axes[1].plot(submission["Date"], submission["COGS"] / 1e6,
             label="V8.26 COGS Forecast", color="darkorange", linewidth=1.2)
axes[1].set_title("COGS Forecast — V8.26 Zero-Sum Extinction", fontsize=13)
axes[1].set_ylabel("COGS (M VND)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUT / "v8_26_forecast_plot.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved.")

## SHAP Explainability

In [ ]:
# SHAP feature importance for Revenue model
import shap

# Use a 500-row subsample for speed
rng = np.random.default_rng(SEED)
sample_idx = rng.choice(len(X_rev), size=min(500, len(X_rev)), replace=False)
X_sample = X_rev[sample_idx]

explainer   = shap.TreeExplainer(rev_model)
shap_values = explainer.shap_values(X_sample)

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(
    shap_values,
    features=X_sample,
    feature_names=FEATURE_COLS,
    show=False,
    max_display=20,
)
plt.title("SHAP Summary — Revenue LightGBM", fontsize=13)
plt.tight_layout()
plt.savefig(OUT / "v8_26_shap_summary.png", dpi=120, bbox_inches="tight")
plt.show()

# Feature importance DataFrame
mean_abs_shap = np.abs(shap_values).mean(axis=0)
fi_df = pd.DataFrame({
    "feature"       : FEATURE_COLS,
    "mean_abs_shap" : mean_abs_shap,
}).sort_values("mean_abs_shap", ascending=False)

fi_df.to_csv(OUT / "v8_26_shap_importance.csv", index=False)
print("Top-15 features by mean |SHAP|:")
print(fi_df.head(15).to_string(index=False))

In [ ]:
# LightGBM native feature importance (gain)
fi_gain = pd.DataFrame({
    "feature": FEATURE_COLS,
    "importance_gain": rev_model.feature_importances_,
}).sort_values("importance_gain", ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(
    data=fi_gain.head(20),
    y="feature",
    x="importance_gain",
    orient="h",
    ax=ax,
)
ax.set_title("LightGBM Feature Importance (Gain) — Revenue Model", fontsize=12)
ax.set_xlabel("Importance (Gain)")
plt.tight_layout()
plt.savefig(OUT / "v8_26_feature_importance.png", dpi=120, bbox_inches="tight")
plt.show()
print("Feature importance plot saved.")

In [ ]:
# Pipeline summary
print("=" * 60)
print("V8.26 PIPELINE SUMMARY")
print("=" * 60)

stages = [
    ("A", "STL + LightGBM base",                         stage_a["Revenue"].mean()),
    ("F", f"Flat ×{LADDER_MULT} anchor (v8_12)",          v8_12["Revenue"].mean()),
    ("D", "L-BFGS-B smooth-MAE calibration (v8_14)",     v8_14["Revenue"].mean()),
    ("H", f"Amplify Δ α={ALPHA} (v8_20)",                v8_20["Revenue"].mean()),
    ("G", "Segmented month Ridge (v8_21)",                v8_21["Revenue"].mean()),
    ("J", f"Zero-Sum Extinction r={EXTINCTION_RATE} (v8_26)", submission["Revenue"].mean()),
]
print(f"{'Stage':<5} {'Description':<45} {'Mean Revenue':>15}")
print("-" * 65)
for stage, desc, mean_rev in stages:
    print(f"[{stage}]   {desc:<45} {mean_rev:>15,.0f}")

print("\nDead SKUs identified (Stage I):")
for pid in dead_product_ids:
    row = dead_skus[dead_skus["product_id"] == pid]
    name = row["product_name"].values[0] if "product_name" in row.columns and len(row) else f"ID={pid}"
    yoy  = float(row["yoy_2021_2022"].values[0]) if "yoy_2021_2022" in row.columns and len(row) else float("nan")
    print(f"  product_id={pid}  {name}  YoY={yoy*100:.1f}%")

print(f"\nGlobal rescale factor K_rev = {K_rev:.7f}")
print(f"Output: {OUT / 'v8_26_zero_sum_extinction.csv'}")
print("=" * 60)